In [39]:
#!pip install emoji
import pandas as pd

file = "Random Tweets.csv"   #file path

data = pd.read_csv(file, encoding='latin-1') #load data

print(data.columns)
print(data.head())

Index(['Unnamed: 0', 'created_at_tweet', 'full_text', 'retweet_count',
       'favorite_count', 'reply_count', 'location'],
      dtype='object')
   Unnamed: 0                created_at_tweet  \
0           0  Wed Jul 03 07:20:15 +0000 2019   
1           1  Fri Jul 02 02:50:53 +0000 2021   
2           2  Fri Jul 02 12:12:53 +0000 2021   
3           3  Tue Oct 15 22:29:16 +0000 2019   
4           4  Wed May 13 04:34:31 +0000 2020   

                                           full_text retweet_count  \
0  ØªÛØ±Ø§ ÙÛÚØ± ÙÛØ±Ø§ ÙÛÚØ± ÙÙØ§Ø² Ø...           226   
1  Happy birthday to my brother n boss , May you ...            54   
2                                       â¤ï¸â¤ï¸            50   
3  `suspicious Â°jikook au jimin'in yaÅadÄ±ÄÄ± ...           522   
4  Speaking of @SpiderMan... ð   https://t.co/...            74   

  favorite_count reply_count                 location  
0           1097         127         Punjab, Pakistan  
1            273          

C:\Users\hp\AppData\Local\Temp\ipykernel_4452\3913366154.py:6: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file, encoding='latin-1') #load data


In [40]:
import re

def clean(t):
    t = str(t).lower()
    t = re.sub(r'\brt\b', ' ', t)               # remove RT
    t = re.sub(r'http\S+|www\.\S+', ' ', t)     # remove websites
    t = re.sub(r'@\w+', ' ', t)                 # remove mentions
    t = re.sub(r'#\w+', ' ', t)                 # remove hashtags
    t = re.sub(r'[^a-z\s]', ' ', t)             # remove punctuation, digits
    t = re.sub(r'[^\x00-\x7F]+', ' ', t)        # remove emojis / non-ASCII
    t = re.sub(r'\s+', ' ', t).strip()
    return t

data['clean'] = data['full_text'].apply(clean) 
#runs clean on each tweet, one at a time.
#The results are stored in a new column called clean.
    #The original full_text is unchanged.

data = data[data['clean'].str.len() > 0] #character count
#for sanity check

print(len(data))
print(data['clean'].head(10))

143297
1     happy birthday to my brother n boss may you ha...
3     suspicious jikook au jimin in ya ad kasabada a...
4                                           speaking of
5                                         alexa skip to
6       jb tm moon lover ho gy to sooraj to tappy ga na
7                                     alexa skip to now
8     but this was one of my first shoots about year...
9                                trying phoneix in ping
10                                cant sleep so why not
11                               ayy gonna miss you too
Name: clean, dtype: object


In [ ]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.tokenize import word_tokenize

#tokenize indiviual tweets
tweets = []
for tweet in data['clean']: tweets.append(word_tokenize(tweet))
   

In [42]:
samplesize = int(len(tweets) *0.2)
trainingsize = int(len(tweets) *0.8)
print("Sample Size: ", samplesize)
print("Training Size: ", trainingsize)
print("Total Size: ", len(tweets))

Sample Size:  28659
Training Size:  114637
Total Size:  143297


In [43]:
#Create Bigram Language Model
train, vocab = padded_everygram_pipeline(2, tweets[0:trainingsize]) 

In [44]:
#Liklihood Model
from nltk.lm import MLE
lm = MLE(2)
lm.fit(train, vocab) #Train the Maximum Likelihood Model

In [45]:
#preparing test data for evalutation

from nltk.util import bigrams
from nltk.lm.preprocessing import pad_both_ends

tests = tweets[trainingsize:]
test_bigrams = []

for t in tests:
    test_bigrams.extend(bigrams(pad_both_ends(t, n=2)))


In [46]:
# evaluation
import math
total = 0
n = 0
for w1, w2 in test_bigrams:
    p = lm.score(w2, [w1])
    if p > 0:
        total += math.log2(p)
        n += 1

print(n)
print(2 ** (-total / n))

316932
92.00887254306501


In [ ]:
print(lm.score('is', ['pakistan'])) 
print(lm.score('pakistan')) 

0.05215812494841958
0.00605638632117391
165.11496244945118
0.0


In [55]:
words = lm.generate(20, random_seed=42)
sentence = ''
for w in words:
    if w == '<s>':
        sentence += ". "
    elif w == '</s>':
        break
    else:
        sentence += w + ' '

print(sentence)

of a fearless and silah fi re all our awaam in pakistan 
